# 05 · Đánh giá tổng hợp B0 / B1 / Proposed + Excel 3 sheet gốc, tổng 6 sheet sau khi thêm 3 seed

Cùng 857 bản thu FLEURS test, cùng sacreBLEU 2.4.3. WER/CER chỉ cho hệ có transcript tiếng Việt (cascade); B1 không có.
Xuất prediction theo schema chung (`reports/danh_gia/predictions_<model>.jsonl` + 1 file CSV gộp) và workbook
`reports/danh_gia/bang_ket_qua.xlsx` (Baseline · Proposed & Ablation · Error & Feasibility). Mục nào chưa chạy ghi **NOT RUN**.

In [1]:
import os, sys, json, csv, subprocess, time, shutil, statistics, re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# Thư mục thực nghiệm (có scripts/, data/, checkpoints/): task3/ – notebook nằm ở task3/notebooks/
T3 = next(c for c in [Path.cwd().parent, Path.cwd(), Path(r"D:\CAIDCM\teamwork\task3")] if (c / "scripts" / "mt" / "train_lora.py").exists())
os.chdir(T3); TW = T3.parent
PY = sys.executable
ENV = dict(os.environ, HF_HUB_CACHE=str(T3 / "hf_home" / "hub"), HF_HUB_OFFLINE="1", HF_HUB_DISABLE_TELEMETRY="1",
           PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
MTREV = "ae7baa85da07dbe8e23ac26a9f5ef560c17e2138"
BO_QUA = ("Warning", "warnings.warn", "warn(", "forced_decoder_ids", "attention mask", "fget.__get__", "TypedStorage")

def chay(*lenh, log=None):
    """Chạy 1 script của project (cùng venv), in log từng dòng, ghi thêm vào file log nếu có; lỗi thì dừng ô."""
    t0 = time.time()
    p = subprocess.Popen([PY, "-X", "utf8", *map(str, lenh)], cwd=T3, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace")
    fl = open(T3 / log, "a", encoding="utf-8") if log else None
    for d in p.stdout:
        if fl: fl.write(d)
        if not any(x in d for x in BO_QUA): print(d, end="")
    p.wait(); fl and fl.close()
    print(f"--- mã thoát {p.returncode} · {time.time() - t0:.0f} s")
    if p.returncode: raise RuntimeError(f"lệnh lỗi: {lenh}")

J = lambda p: json.loads(Path(p).read_text(encoding="utf-8"))
def mt(v, s):
    p = T3 / "runs/mt" / v / s / "score.json"
    return J(p) if p.exists() else None
vn = lambda x, d=2: "–" if x is None else f"{float(x):.{d}f}".replace(".", ",")
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})
import torch
print("Thư mục:", T3, "| Python", sys.version.split()[0], "| torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG")

Thư mục: D:\CAIDCM\teamwork\task3 | Python 3.11.1 | torch 2.0.1+cu117 | GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 1. Prediction theo schema chung (chuyển từ kết quả đã chạy, không chạy lại model)

In [2]:
OUTD = T3 / "reports/danh_gia"; OUTD.mkdir(parents=True, exist_ok=True)
man = {json.loads(l)["id"]: json.loads(l) for l in open("runs/manifests/asr_fleurs_test.jsonl", encoding="utf-8")}
en = {json.loads(l)["id"]: json.loads(l)["ref"] for l in open("runs/mt_data/fleurs_test_clean.jsonl", encoding="utf-8")}
asr = {s: {json.loads(l)["id"]: json.loads(l)["prediction"] for l in open(f"runs/asr/kiet_t2_{s}_fleurs_test/predictions.jsonl", encoding="utf-8")}
       for s in ("phowhisper-small", "whisper-small")}
chon = J("reports/ablation/chon_ban_cuoi.json")["ban_chon"]
HE = {"B0": ("B0_cascade_PhoWhisper+VinAI", "B0", "fleurs_test_asr_pho", "phowhisper-small"),
      "B0_wsp": ("cascade_WhisperSmall+VinAI", "B0", "fleurs_test_asr_wsp", "whisper-small"),
      **{v: (f"Proposed_PhoWhisper+VinAI+LoRA_{v}", v, "fleurs_test_asr_pho", "phowhisper-small") for v in ("V1", "V2", "V3", "V4", "A_r8", "A_r32", "A_aug")}}
b1 = {r["sample_id"] + ".wav": r for r in (json.loads(l) for l in open("runs/b1/predictions_b1.jsonl", encoding="utf-8"))}
tat = []
for k, (mid, v, s, a) in HE.items():
    hyp = {json.loads(l)["id"]: json.loads(l)["hyp"] for l in open(f"runs/mt/{v}/{s}/hyp.jsonl", encoding="utf-8")}
    R = [{"sample_id": i.removesuffix(".wav"), "split": "test", "model_id": mid, "reference_vi": man[i]["reference"], "reference_en": en[i], "pred_vi": asr[a][i],
          "pred_en": hyp[i], "latency_s": None, "audio_duration_s": man[i]["duration_s"], "status": "ok"} for i in man]
    tat += R
    with open(OUTD / f"predictions_{k}.jsonl", "w", encoding="utf-8") as f:
        for r in R: f.write(json.dumps(r, ensure_ascii=False) + "\n")
R = [{k_: b1[i][k_] for k_ in ("sample_id", "split", "model_id", "reference_vi", "reference_en", "pred_vi", "pred_en", "latency_s", "audio_duration_s", "status")} for i in man]
tat += R
with open(OUTD / "predictions_B1.jsonl", "w", encoding="utf-8") as f:
    for r in R: f.write(json.dumps(r, ensure_ascii=False) + "\n")
df = pd.DataFrame(tat); df.to_csv(OUTD / "predictions_tat_ca.csv", index=False, encoding="utf-8-sig")
print(df.groupby("model_id").size().to_string()); print("latency_s mỗi mẫu: chỉ B1 đo khi chạy; cascade đo tốc độ riêng (reports/toc_do/lap_V4) → NULL trong file này")

model_id
B0_cascade_PhoWhisper+VinAI             857
B1_whisper-small_translate@973afd24     857
Proposed_PhoWhisper+VinAI+LoRA_A_aug    857
Proposed_PhoWhisper+VinAI+LoRA_A_r32    857
Proposed_PhoWhisper+VinAI+LoRA_A_r8     857
Proposed_PhoWhisper+VinAI+LoRA_V1       857
Proposed_PhoWhisper+VinAI+LoRA_V2       857
Proposed_PhoWhisper+VinAI+LoRA_V3       857
Proposed_PhoWhisper+VinAI+LoRA_V4       857
cascade_WhisperSmall+VinAI              857
latency_s mỗi mẫu: chỉ B1 đo khi chạy; cascade đo tốc độ riêng (reports/toc_do/lap_V4) → NULL trong file này


## 2. Bảng so sánh chung trên test

In [3]:
import statistics as st
rs = {r["run_id"]: r for r in csv.DictReader(open("reports/asr/results_task2.csv", encoding="utf-8-sig"))}
PHIEN = next(d for d in ("lap_V2", "lap_V4") if (T3 / "reports/toc_do" / d / "toc_do_V2_lan3.json").exists() or d == "lap_V4")  # cùng 1 phiên cho mọi hệ
lap = lambda v: [J(p) for p in sorted((T3 / "reports/toc_do" / PHIEN).glob(f"toc_do_{v}_lan*.json"))]
print("Phiên đo tốc độ:", PHIEN)
def td(v):
    L = lap(v); return (st.median(x["RTF_total"] for x in L), st.median(x["latency_p95_s"] for x in L), max(x["gpu_peak_allocated_bytes"] for x in L) / 2**30) if L else (None, None, None)
b1s = mt("B1_whisper-small_translate", "fleurs_test_audio")
siz = lambda p: sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file())
hf = T3 / "hf_home/hub"
SZ = {"pho": siz(hf / "models--vinai--PhoWhisper-small/blobs"), "wsp": siz(hf / "models--openai--whisper-small/blobs"), "mt": siz(hf / "models--vinai--vinai-translate-vi2en-v2/blobs")}
rows = [{"hệ": "B0 · PhoWhisper → VinAI gốc", "WER FLEURS (%)": rs["kiet_t2_phowhisper-small_fleurs_test"]["WER_percent"], "BLEU test": mt("B0", "fleurs_test_asr_pho")["BLEU"],
         "chrF++": mt("B0", "fleurs_test_asr_pho")["chrF++"], "BLEU dịch sạch": mt("B0", "fleurs_test_clean")["BLEU"], "RTF": td("B0")[0], "p95 (s)": td("B0")[1]},
        {"hệ": "Whisper-small → VinAI gốc (cascade)", "WER FLEURS (%)": rs["kiet_t2_whisper-small_fleurs_test"]["WER_percent"], "BLEU test": mt("B0", "fleurs_test_asr_wsp")["BLEU"],
         "chrF++": mt("B0", "fleurs_test_asr_wsp")["chrF++"], "BLEU dịch sạch": "–", "RTF": "NOT RUN", "p95 (s)": "NOT RUN"},
        {"hệ": "B1 · Whisper-small translate (end-to-end)", "WER FLEURS (%)": "không áp dụng", "BLEU test": b1s["BLEU"], "chrF++": b1s["chrF++"], "BLEU dịch sạch": "không áp dụng",
         "RTF": f"{b1s['RTF_tong']} (cách đo khác)", "p95 (s)": b1s["latency_p95_s"]}]
for v in ("V1", "V2", "V3", "V4", "A_r8", "A_r32", "A_aug"):
    rows.append({"hệ": f"Proposed {v}" + (" ★ chọn theo dev" if v == chon else ""), "WER FLEURS (%)": rs["kiet_t2_phowhisper-small_fleurs_test"]["WER_percent"],
                 "BLEU test": mt(v, "fleurs_test_asr_pho")["BLEU"], "chrF++": mt(v, "fleurs_test_asr_pho")["chrF++"], "BLEU dịch sạch": mt(v, "fleurs_test_clean")["BLEU"],
                 "RTF": td(v)[0] if td(v)[0] else "NOT RUN", "p95 (s)": td(v)[1] if td(v)[1] else "NOT RUN"})
TONG = pd.DataFrame(rows); display(TONG)
print("Kích thước checkpoint (GB): PhoWhisper", round(SZ["pho"] / 1e9, 2), "· Whisper-small", round(SZ["wsp"] / 1e9, 2), "· VinAI", round(SZ["mt"] / 1e9, 2), "· adapter 0,035")

Phiên đo tốc độ: lap_V2


,hệ,WER FLEURS (%),BLEU test,chrF++,BLEU dịch sạch,RTF,p95 (s)
0,B0 · PhoWhisper → VinAI gốc,13.68,22.69,49.38,38.27,0.115,2.24
1,Whisper-small → VinAI gốc (cascade),21.24,18.58,44.62,–,NOT RUN,NOT RUN
2,B1 · Whisper-small translate (end-to-end),không áp dụng,3.80,25.45,không áp dụng,0.137 (cách đo khác),8.196
3,Proposed V1,13.68,27.49,52.50,36.21,NOT RUN,NOT RUN
4,Proposed V2 ★ chọn theo dev,13.68,27.47,52.98,38.09,0.113,2.24
5,Proposed V3,13.68,27.33,52.25,36.76,NOT RUN,NOT RUN
6,Proposed V4,13.68,27.37,52.51,38.11,0.11,2.14
7,Proposed A_r8,13.68,27.26,52.82,37.96,NOT RUN,NOT RUN
8,Proposed A_r32,13.68,27.62,52.93,37.96,NOT RUN,NOT RUN
9,Proposed A_aug,13.68,25.49,51.61,36.65,NOT RUN,NOT RUN


Kích thước checkpoint (GB): PhoWhisper 0.0 · Whisper-small 0.0 · VinAI 0.0 · adapter 0,035


## 3. Workbook 3 sheet gốc, tổng 6 sheet – `reports/danh_gia/bang_ket_qua.xlsx`

In [4]:
pc = pd.read_csv("reports/asr/paper_comparison.csv", encoding="utf-8-sig")
S1 = pd.DataFrame([
  {"baseline": "PhoWhisper-small (ASR)", "dataset": "VIVOS test (760)", "metric": "WER %", "paper": 6.33, "nhóm": float(rs["kiet_t2_phowhisper-small_vivos_test"]["WER_percent"]),
   "ghi chú": "cùng dataset; paper PhoWhisper Table 2", "nguồn": "reports/asr/results_task2.csv"},
  {"baseline": "Whisper-small (ASR)", "dataset": "FLEURS vi_vn test (857)", "metric": "WER %", "paper": 21.2, "nhóm": float(rs["kiet_t2_whisper-small_fleurs_test"]["WER_percent"]),
   "ghi chú": "cùng dataset; paper Whisper Table 11", "nguồn": "reports/asr/results_task2.csv"},
  {"baseline": "VinAI Translate vi2en (MT gốc)", "dataset": "PhoMT test (19.151)", "metric": "BLEU", "paper": 40.42, "nhóm": mt("B0", "phomt_test")["BLEU"],
   "ghi chú": "cùng dataset; paper Interspeech 2022 Table 1", "nguồn": "runs/mt/B0/phomt_test/score.json"},
  {"baseline": "B0 cascade PhoWhisper → VinAI", "dataset": "FLEURS vi_vn test (857)", "metric": "BLEU", "paper": "NR", "nhóm": mt("B0", "fleurs_test_asr_pho")["BLEU"],
   "ghi chú": "paper không báo cascade; dịch sạch cùng tập: " + str(mt("B0", "fleurs_test_clean")["BLEU"]), "nguồn": "runs/mt/B0/fleurs_test_asr_pho/score.json"},
  {"baseline": "B1 Whisper-small translate (end-to-end)", "dataset": "FLEURS vi_vn test (857)", "metric": "BLEU", "paper": "NOT RUN (chưa tra bảng dịch FLEURS của paper Whisper)", "nhóm": b1s["BLEU"],
   "ghi chú": "baseline đối chứng theo proposal Task 1; không nhận là tái lập paper", "nguồn": "runs/mt/B1_whisper-small_translate/fleurs_test_audio/score.json"}])
S2 = pd.read_csv("reports/ablation/bang_ablation.csv", encoding="utf-8-sig").assign(nguồn="runs/mt/<bản>/<tập>/score.json; reports/mt/bootstrap_V2_*.json; chọn: reports/ablation/chon_ban_cuoi.json")
grp = {"all": "tất cả", "so": "có chữ số", "ten": "có tên riêng", "dai": "lời ASR > 40 từ", "asr_xau": "ASR sai nhiều", "con_lai": "còn lại"}
e3 = []
for r in csv.DictReader(open("reports/asr/strata.csv", encoding="utf-8-sig")):
    if r["run_id"] == "kiet_t2_phowhisper-small_fleurs_test":
        e3.append({"loại": "WER ASR theo nhóm (test)", "nhóm": f"{r['group_type']}={r['group_value']}", "n": r["n_recordings"], "giá trị": r["WER_percent"], "đơn vị": "WER %", "nguồn": "reports/asr/strata.csv"})
fv = chon if (T3 / f"reports/mt/failure_{chon}_dev.json").exists() else "V4"
g = J(f"reports/mt/failure_{fv}_dev.json")["groups"]
for k, t in grp.items():
    e3.append({"loại": f"BLEU dev theo nhóm câu: B0 → {fv}", "nhóm": t, "n": g[k]["n"], "giá trị": f"{g[k]['B0'][0]} → {g[k][fv][0]}", "đơn vị": "BLEU", "nguồn": f"reports/mt/failure_{fv}_dev.json"})
for v in ("B0", "V2", "V3", "V4"):
    rtf, p95, vr = td(v)
    if rtf is not None:
        e3.append({"loại": "Tốc độ (3 lần đo xen kẽ, 100 bản thu)", "nhóm": v, "n": 100, "giá trị": f"RTF {rtf} · p95 {p95} s · VRAM {vr:.2f} GiB", "đơn vị": "", "nguồn": f"reports/toc_do/{PHIEN}/"})
e3.append({"loại": "Tốc độ B1", "nhóm": "B1", "n": 857, "giá trị": f"RTF {b1s['RTF_tong']} · p95 {b1s['latency_p95_s']} s", "đơn vị": "", "nguồn": "runs/mt/B1_whisper-small_translate/fleurs_test_audio/score.json"})
he = next((T3 / f"reports/{d}/ket_qua.json" for d in ("he_thong_V2", "he_thong") if (T3 / f"reports/{d}/ket_qua.json").exists()), T3 / "x")
if he.exists():
    k_ = J(he)
    for c_ in ("truoc_0411", "proposed"):
        q = k_[c_]["tat_ca"]; e3.append({"loại": f"Cả hệ sản phẩm ({k_['mo_hinh'].split('LoRA ')[-1]} + tiền/hậu xử lý)", "nhóm": c_, "n": q["n"], "giá trị": f"WER {q['WER_asr']} · BLEU {q['BLEU']} · chrF++ {q['chrF++']} · RTF {q['RTF']}", "đơn vị": "", "nguồn": str(he.relative_to(T3))})
e3.append({"loại": "Kích thước", "nhóm": "PhoWhisper + VinAI + adapter", "n": "", "giá trị": round((SZ["pho"] + SZ["mt"]) / 1e9 + 0.035, 2), "đơn vị": "GB", "nguồn": "hf_home/hub"})
S3 = pd.DataFrame(e3)
X = OUTD / "bang_ket_qua.xlsx"
with pd.ExcelWriter(X, engine="openpyxl") as w:
    S1.to_excel(w, sheet_name="Baseline Comparison", index=False); S2.to_excel(w, sheet_name="Proposed & Ablation", index=False); S3.to_excel(w, sheet_name="Error & Feasibility", index=False)
    TONG.to_excel(w, sheet_name="So sanh chung (test)", index=False)
from openpyxl import load_workbook
wb = load_workbook(X)
for ws in wb.worksheets:
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = min(60, max(10, max(len(str(c.value or "")) for c in col) + 2))
wb.save(X)
print(X, "| sheet:", wb.sheetnames); display(S1); display(S3.head(12))

D:\CAIDCM\teamwork\task3\reports\danh_gia\bang_ket_qua.xlsx | sheet: ['Baseline Comparison', 'Proposed & Ablation', 'Error & Feasibility', 'So sanh chung (test)']


,baseline,dataset,metric,paper,nhóm,ghi chú,nguồn
0,PhoWhisper-small (ASR),VIVOS test (760),WER %,6.33,6.24,cùng dataset; paper PhoWhisper Table 2,reports/asr/results_task2.csv
1,Whisper-small (ASR),FLEURS vi_vn test (857),WER %,21.2,21.24,cùng dataset; paper Whisper Table 11,reports/asr/results_task2.csv
2,VinAI Translate vi2en (MT gốc),PhoMT test (19.151),BLEU,40.42,40.79,cùng dataset; paper Interspeech 2022 Table 1,runs/mt/B0/phomt_test/score.json
3,B0 cascade PhoWhisper → VinAI,FLEURS vi_vn test (857),BLEU,NR,22.69,paper không báo cascade; dịch sạch cùng tập: 38.27,runs/mt/B0/fleurs_test_asr_pho/score.json
4,B1 Whisper-small translate (end-to-end),FLEURS vi_vn test (857),BLEU,NOT RUN (chưa tra bảng dịch FLEURS của paper Whisper),3.80,baseline đối chứng theo proposal Task 1; không nhận là tái lập paper,runs/mt/B1_whisper-small_translate/fleurs_test_audio/score.json


,loại,nhóm,n,giá trị,đơn vị,nguồn
0,WER ASR theo nhóm (test),duration_s=<=5,2,3.85,WER %,reports/asr/strata.csv
1,WER ASR theo nhóm (test),duration_s=>15_to_30,209,15.24,WER %,reports/asr/strata.csv
2,WER ASR theo nhóm (test),duration_s=>30,7,94.7,WER %,reports/asr/strata.csv
3,WER ASR theo nhóm (test),duration_s=>5_to_15,639,10.58,WER %,reports/asr/strata.csv
4,WER ASR theo nhóm (test),name_or_number=name,552,16.52,WER %,reports/asr/strata.csv
5,WER ASR theo nhóm (test),name_or_number=neither,278,5.89,WER %,reports/asr/strata.csv
6,WER ASR theo nhóm (test),name_or_number=number,171,27.43,WER %,reports/asr/strata.csv
7,BLEU dev theo nhóm câu: B0 → V2,tất cả,361,22.96 → 26.45,BLEU,reports/mt/failure_V2_dev.json
8,BLEU dev theo nhóm câu: B0 → V2,có chữ số,44,10.86 → 18.78,BLEU,reports/mt/failure_V2_dev.json
9,BLEU dev theo nhóm câu: B0 → V2,có tên riêng,210,20.3 → 25.28,BLEU,reports/mt/failure_V2_dev.json
